# **Mô hình YOLOv8**

### **1. Thiết lập môi trường và Cấu hình ban đầu**

#### **1.1. Import các thư viện và thiết lập môi trường**
Cài đặt và import các thư viện cần thiết, mount Google Drive để lưu và đọc dữ liệu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics opencv-python pandas pyyaml -q

import os, json, time, shutil, zipfile, gc
from pathlib import Path
from datetime import datetime

import yaml
import pandas as pd
import torch
from ultralytics import YOLO
from IPython.display import Image, display

#### **1.2. Cấu hình đường dẫn và tham số mô hình (Hyperparameters)**
Khai báo đường dẫn dữ liệu, các siêu tham số (hyperparameters) như EPOCHS, BATCH_SIZE, IMAGE_SIZE và thiết lập thiết bị chạy (CPU/GPU).

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/ObjectDetection/voc_traffic_5classes_full.zip"

DATASET_ROOT = Path("/content/voc_traffic_5classes")
SUBSET_ROOT = DATASET_ROOT / "subset_3000_500_1000"
YOLO_DIR = SUBSET_ROOT / "yolo"
DATA_YAML = YOLO_DIR / "data.yaml"

OUTPUT_ROOT = Path("/content/model_results")
MODEL_NAME = "yolov8s"
MODEL_DISPLAY_NAME = "YOLOv8s"
RUN_NAME = "yolov8s_train"

CLASSES = ["person", "car", "bus", "bicycle", "motorbike"]

EPOCHS = 30
BATCH_SIZE = 16
IMAGE_SIZE = 640
BATCH_PREDICT = 32

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

MODEL_RESULT_DIR = OUTPUT_ROOT / MODEL_NAME
CONFIG_DIR = MODEL_RESULT_DIR / "config"
CHECKPOINT_DIR = MODEL_RESULT_DIR / "checkpoints"
METRICS_DIR = MODEL_RESULT_DIR / "metrics"
PREDICTIONS_DIR = MODEL_RESULT_DIR / "predictions"
SAMPLE_PRED_DIR = PREDICTIONS_DIR / "sample_predictions"
LOGS_DIR = MODEL_RESULT_DIR / "logs"

TRAINING_LOG_PATH = LOGS_DIR / "training_log.txt"
README_PATH = MODEL_RESULT_DIR / "README.md"
ZIP_OUTPUT_PATH = Path("/content/drive/MyDrive/ObjectDetection/yolov8s_results.zip")

#### **1.3. Tạo thư mục lưu kết quả**
Tạo các thư mục để lưu checkpoint, metrics, log và dự đoán mẫu.

In [ ]:
if MODEL_RESULT_DIR.exists():
    shutil.rmtree(MODEL_RESULT_DIR)

for d in [CONFIG_DIR, CHECKPOINT_DIR, METRICS_DIR, PREDICTIONS_DIR, SAMPLE_PRED_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def write_log(text):
    print(text)
    with open(TRAINING_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(str(text) + "\n")

with open(TRAINING_LOG_PATH, "w", encoding="utf-8") as f:
    f.write("YOLOv8s Training Log\n")
    f.write("=" * 50 + "\n")

write_log(f"Timestamp: {datetime.now()}")
write_log(f"ZIP_PATH: {ZIP_PATH}")
write_log(f"GPU: {GPU_NAME}")

### **2. Chuẩn bị dữ liệu (Data Preparation)**

#### **2.1. Giải nén dữ liệu VOC**
Giải nén tập dữ liệu VOC vào thư mục tiêu chuẩn trên Colab.

In [ ]:
if DATASET_ROOT.exists():
    write_log(f"Removing old dataset folder: {DATASET_ROOT}")
    shutil.rmtree(DATASET_ROOT)

DATASET_ROOT.mkdir(parents=True, exist_ok=True)

write_log("Unzipping dataset into /content/voc_traffic_5classes ...")

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATASET_ROOT)

write_log("Dataset unzipped.")

if not YOLO_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy YOLO_DIR: {YOLO_DIR}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Không tìm thấy DATA_YAML: {DATA_YAML}")

#### **2.2. Cập nhật file cấu hình dữ liệu (data.yaml)**
Chỉnh sửa lại đường dẫn trong file `data.yaml` cho phù hợp với thư mục hiện tại.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["path"] = str(YOLO_DIR)

if "names" not in data_yaml:
    data_yaml["names"] = {i: name for i, name in enumerate(CLASSES)}

FIXED_DATA_YAML = CONFIG_DIR / "data.yaml"

with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

write_log("Fixed data.yaml saved.")

#### **2.3. Đếm và kiểm tra số lượng ảnh (Train/Val/Test)**
Kiểm tra số lượng ảnh trong các tập train, val, test.

In [ ]:
def count_images(split):
    with open(FIXED_DATA_YAML, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    img_dir = YOLO_DIR / cfg[split]
    total = 0

    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        total += len(list(img_dir.glob(ext)))

    return total

TRAIN_IMAGES = count_images("train")
VAL_IMAGES = count_images("val")
TEST_IMAGES = count_images("test")

write_log(f"Train images: {TRAIN_IMAGES}")
write_log(f"Val images: {VAL_IMAGES}")
write_log(f"Test images: {TEST_IMAGES}")

### **3. Huấn luyện mô hình (Training)**

#### **3.1. Lưu cấu hình huấn luyện (training_config.json)**
Lưu lại các thông số cấu hình vào file `training_config.json`.

In [ ]:
training_config = {
    "model_name": "YOLOv8s",
    "model_type": "One-stage object detector",
    "dataset": "VOC0712 traffic subset 3000/500/1000",
    "dataset_path": str(DATA_YAML),
    "fixed_data_yaml": str(FIXED_DATA_YAML),
    "classes": CLASSES,
    "num_classes": 5,
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "test_images": TEST_IMAGES,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "learning_rate": "default_ultralytics",
    "weight_decay": "default_ultralytics",
    "optimizer": "auto/default_ultralytics",
    "scheduler": "default_ultralytics",
    "pretrained": True,
    "freeze_backbone": False,
    "device": DEVICE_NAME,
    "gpu": GPU_NAME
}

with open(CONFIG_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=4, ensure_ascii=False)

write_log("training_config.json saved.")

#### **3.2. Huấn luyện mô hình YOLOv8s**
Bắt đầu quá trình huấn luyện và lưu lại checkpoint tốt nhất.

In [ ]:
write_log("Start training YOLOv8s...")

model = YOLO("yolov8s.pt")
train_start = time.time()

model.train(
    data=str(FIXED_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    project=str(LOGS_DIR),
    name=RUN_NAME,
    patience=10,
    save=True,
    plots=True
)

train_end = time.time()
total_train_time = train_end - train_start
avg_epoch_time = total_train_time / EPOCHS

write_log(f"Training finished. Total time: {total_train_time:.2f} seconds")
write_log(f"Average epoch time: {avg_epoch_time:.2f} seconds")

RUN_DIR = LOGS_DIR / RUN_NAME
YOLO_BEST = RUN_DIR / "weights" / "best.pt"
YOLO_LAST = RUN_DIR / "weights" / "last.pt"

if not YOLO_BEST.exists():
    raise FileNotFoundError(f"Không tìm thấy best.pt: {YOLO_BEST}")

if not YOLO_LAST.exists():
    raise FileNotFoundError(f"Không tìm thấy last.pt: {YOLO_LAST}")

shutil.copy(YOLO_BEST, CHECKPOINT_DIR / "best.pt")
shutil.copy(YOLO_LAST, CHECKPOINT_DIR / "last.pt")

BEST_MODEL_PATH = CHECKPOINT_DIR / "best.pt"

#### **3.3. Trích xuất lịch sử huấn luyện (Training History)**
Trích xuất và lưu lịch sử huấn luyện từ file `results.csv`.

In [ ]:
results_csv = RUN_DIR / "results.csv"
training_history_path = METRICS_DIR / "training_history.csv"

if results_csv.exists():
    hist = pd.read_csv(results_csv)
    hist.columns = [c.strip() for c in hist.columns]

    train_loss_cols = [c for c in hist.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss_cols = [c for c in hist.columns if "val" in c.lower() and "loss" in c.lower()]

    history_out = pd.DataFrame()
    history_out["epoch"] = hist["epoch"] if "epoch" in hist.columns else range(1, len(hist) + 1)
    history_out["train_loss"] = hist[train_loss_cols].sum(axis=1) if train_loss_cols else None
    history_out["val_loss"] = hist[val_loss_cols].sum(axis=1) if val_loss_cols else None
    history_out["learning_rate"] = hist["lr/pg0"] if "lr/pg0" in hist.columns else None
    history_out["epoch_time_seconds"] = avg_epoch_time
    history_out["precision"] = hist["metrics/precision(B)"] if "metrics/precision(B)" in hist.columns else None
    history_out["recall"] = hist["metrics/recall(B)"] if "metrics/recall(B)" in hist.columns else None
    history_out["mAP50"] = hist["metrics/mAP50(B)"] if "metrics/mAP50(B)" in hist.columns else None
    history_out["mAP50_95"] = hist["metrics/mAP50-95(B)"] if "metrics/mAP50-95(B)" in hist.columns else None
else:
    history_out = pd.DataFrame(columns=[
        "epoch", "train_loss", "val_loss", "learning_rate",
        "epoch_time_seconds", "precision", "recall", "mAP50", "mAP50_95"
    ])

history_out.to_csv(training_history_path, index=False)
write_log("training_history.csv saved.")

### **4. Đánh giá mô hình (Evaluation)**

#### **4.1. Đánh giá trên tập Test (tính toán mAP, Precision, Recall)**
Đánh giá hiệu năng mô hình (mAP, Precision, Recall) trên tập kiểm thử.

In [ ]:
write_log("Start evaluation on test set...")

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(str(BEST_MODEL_PATH))

eval_results = model.val(
    data=str(FIXED_DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(LOGS_DIR),
    name="test_evaluation"
)

mAP50 = float(eval_results.box.map50)
mAP5095 = float(eval_results.box.map)
precision = float(eval_results.box.mp)
recall = float(eval_results.box.mr)

write_log(f"mAP@0.5: {mAP50}")
write_log(f"mAP@0.5:0.95: {mAP5095}")
write_log(f"Precision: {precision}")
write_log(f"Recall: {recall}")

#### **4.2. Chỉ số đánh giá chi tiết theo từng lớp (Per-class Metrics)**
Lưu chi tiết chỉ số mAP, Precision, Recall của từng lớp (Person, Car, Bus...).

In [ ]:
class_maps = eval_results.box.maps

per_class_rows = []
for i, cls in enumerate(CLASSES):
    per_class_rows.append({
        "class": cls,
        "mAP": float(class_maps[i]) if i < len(class_maps) else None,
        "precision": None,
        "recall": None,
        "num_objects": None
    })

pd.DataFrame(per_class_rows).to_csv(METRICS_DIR / "per_class_metrics.csv", index=False)
write_log("per_class_metrics.csv saved.")

#### **4.3. Tính toán tổng số tham số và dung lượng mô hình**
Đếm tổng số lượng tham số, tham số train được và kích thước mô hình (MB).

In [ ]:
total_parameters = sum(p.numel() for p in model.model.parameters())
trainable_parameters = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
model_size_MB = BEST_MODEL_PATH.stat().st_size / (1024 * 1024)

### **5. Suy luận và Dự đoán (Inference & Prediction)**

#### **5.1. Tải danh sách ảnh từ tập Test**
Lấy danh sách các đường dẫn tới file ảnh trong tập test.

In [ ]:
with open(FIXED_DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

test_img_dir = YOLO_DIR / cfg["test"]

test_images = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    test_images.extend(list(test_img_dir.glob(ext)))

test_images = sorted(test_images)

#### **5.2. Đo tốc độ suy luận (Inference Speed & FPS)**
Đo thời gian dự đoán trung bình trên tập test và tính FPS (Frames Per Second).

In [ ]:
if len(test_images) == 0:
    avg_inference_time = None
    fps = None
else:
    speed_start = time.time()

    for i in range(0, len(test_images), BATCH_PREDICT):
        batch_paths = [str(p) for p in test_images[i:i + BATCH_PREDICT]]

        _ = model.predict(
            source=batch_paths,
            imgsz=IMAGE_SIZE,
            conf=0.25,
            device=DEVICE,
            verbose=False,
            stream=False
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    speed_end = time.time()

    total_inference_time = speed_end - speed_start
    avg_inference_time = total_inference_time / len(test_images)
    fps = 1 / avg_inference_time if avg_inference_time > 0 else None

write_log(f"Inference measured on full test set: {len(test_images)} images")
write_log(f"Average inference time: {avg_inference_time}")
write_log(f"FPS: {fps}")

#### **5.3. Dự đoán thử nghiệm và trực quan hóa (trên 10 ảnh mẫu)**
Chạy dự đoán cho 10 ảnh test đầu tiên và xuất kết quả ra file CSV và ảnh minh họa.

In [ ]:
sample_images = sorted(test_images)[:10]
prediction_rows = []

if len(sample_images) > 0:
    pred_results = model.predict(
        source=[str(p) for p in sample_images],
        conf=0.25,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        save=True,
        project=str(PREDICTIONS_DIR),
        name="sample_predictions_tmp",
        verbose=False
    )

    tmp_pred_dir = PREDICTIONS_DIR / "sample_predictions_tmp"

    with open(PREDICTIONS_DIR / "sample_test_images.txt", "w", encoding="utf-8") as f:
        for img_path in sample_images:
            f.write(img_path.name + "\n")

    for img_path, result in zip(sample_images, pred_results):
        src_pred_img = tmp_pred_dir / img_path.name
        dst_pred_img = SAMPLE_PRED_DIR / img_path.name

        if src_pred_img.exists():
            shutil.copy(src_pred_img, dst_pred_img)

        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            prediction_rows.append({
                "image_name": img_path.name,
                "class": CLASSES[cls_id] if cls_id < len(CLASSES) else str(cls_id),
                "confidence": conf,
                "bbox_xmin": x1,
                "bbox_ymin": y1,
                "bbox_xmax": x2,
                "bbox_ymax": y2
            })

pd.DataFrame(prediction_rows).to_csv(PREDICTIONS_DIR / "prediction_examples.csv", index=False)

if (PREDICTIONS_DIR / "sample_predictions_tmp").exists():
    shutil.rmtree(PREDICTIONS_DIR / "sample_predictions_tmp")

write_log("prediction_examples.csv and sample_predictions saved.")

### **6. Tổng hợp kết quả và Lưu trữ**

#### **6.1. Lưu tổng hợp các chỉ số đánh giá (Evaluation Metrics)**
Tổng hợp tất cả các chỉ số hiệu năng và lưu vào file CSV/JSON.

In [ ]:
evaluation_metrics = {
    "model": MODEL_DISPLAY_NAME,
    "dataset": "VOC0712 traffic subset 3000/500/1000",
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "test_images": TEST_IMAGES,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "mAP@0.5_PascalVOC": mAP50,
    "mAP@0.5:0.95_COCO": mAP5095,
    "avg_inference_time_seconds_per_image": avg_inference_time,
    "fps": fps,
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "model_size_MB": model_size_MB,
    "device": DEVICE_NAME
}

pd.DataFrame([evaluation_metrics]).to_csv(METRICS_DIR / "evaluation_metrics.csv", index=False)

with open(METRICS_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_metrics, f, indent=4, ensure_ascii=False)

write_log("evaluation_metrics.csv/json saved.")

#### **6.2. Tự động tạo báo cáo README.md**
Tạo file README tự động tổng hợp thông tin về dữ liệu, mô hình và kết quả.

In [ ]:
readme_content = f"""# YOLOv8s Result

## Dataset
- Dataset: VOC0712 traffic subset 3000/500/1000
- Dataset root: {SUBSET_ROOT}
- YOLO data yaml: {DATA_YAML}
- Fixed training yaml: {FIXED_DATA_YAML}
- Classes: person, car, bus, bicycle, motorbike
- Train/Val/Test: {TRAIN_IMAGES}/{VAL_IMAGES}/{TEST_IMAGES}

## Model
- Architecture: YOLOv8s
- Model type: One-stage object detector
- Pretrained: yolov8s.pt
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Image size: {IMAGE_SIZE}
- Optimizer: auto/default_ultralytics
- Learning rate: default_ultralytics
- Freeze backbone: No
- Device: {DEVICE_NAME}
- GPU: {GPU_NAME}

## Output
- Best checkpoint: checkpoints/best.pt
- Last checkpoint: checkpoints/last.pt
- Main metrics: metrics/evaluation_metrics.csv
- Per-class metrics: metrics/per_class_metrics.csv
- Training history: metrics/training_history.csv
- Prediction examples: predictions/sample_predictions/
- Prediction CSV: predictions/prediction_examples.csv
- Sample image list: predictions/sample_test_images.txt
- Training log: logs/training_log.txt

## Main Results
- mAP@0.5 Pascal VOC: {mAP50}
- mAP@0.5:0.95 COCO: {mAP5095}
- Precision: {precision}
- Recall: {recall}
- Average inference time per image: {avg_inference_time}
- FPS: {fps}
- Inference images used: {len(test_images)}
- Total parameters: {total_parameters}
- Trainable parameters: {trainable_parameters}
- Model size MB: {model_size_MB}
"""

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(readme_content)

write_log("README.md saved.")

#### **6.3. Nén toàn bộ thư mục kết quả (.zip)**
Nén toàn bộ các kết quả lại thành một file zip để dễ dàng tải về.

In [ ]:
if ZIP_OUTPUT_PATH.exists():
    ZIP_OUTPUT_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_OUTPUT_PATH).replace(".zip", ""),
    format="zip",
    root_dir=str(OUTPUT_ROOT),
    base_dir=MODEL_NAME
)

write_log(f"ZIP saved to: {ZIP_OUTPUT_PATH}")

#### **6.4. Kiểm tra lại các file đầu ra (Final Check)**
Hiển thị một số kết quả, bảng chỉ số để kiểm tra tổng quát.

In [ ]:
print("\nDONE.")
print("Output folder:", MODEL_RESULT_DIR)
print("Zip result:", ZIP_OUTPUT_PATH)

!find /content/model_results/yolov8s -maxdepth 4 -type f | sort

display(pd.read_csv(METRICS_DIR / "evaluation_metrics.csv"))
display(pd.read_csv(METRICS_DIR / "per_class_metrics.csv"))
display(pd.read_csv(PREDICTIONS_DIR / "prediction_examples.csv").head())

sample_output_images = list(SAMPLE_PRED_DIR.glob("*"))
if len(sample_output_images) > 0:
    display(Image(filename=str(sample_output_images[0])))